<a href="https://colab.research.google.com/github/Aswathi281099/Generative-Artificial-Intelligence/blob/main/AI_TASK_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TASK 6**

In [10]:
import asyncio
import time
from typing import List
from contextlib import asynccontextmanager

import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

# LangChain & FAISS imports
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Cross-Encoder Reranker
from sentence_transformers import CrossEncoder

# Detect available device (GPU if available, otherwise CPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
class AsyncRAGService:
    def __init__(self):
        self.vector_store = None
        self.reranker = None

    def initialize_models_and_index(self):
        """Loads models and builds the initial FAISS vector index."""
        print(f"[RAG Engine] Loading models on device: {DEVICE}...")

        # Stage 1: Bi-Encoder Embedding Model
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            model_kwargs={"device": DEVICE}
        )

        # Knowledge Base Corpus
        knowledge_corpus = [
            Document(page_content="HNSW (Hierarchical Navigable Small World) is an efficient graph-based algorithm for fast vector search."),
            Document(page_content="FastAPI leverages Python's asyncio event loop to handle concurrent web requests with minimal latency."),
            Document(page_content="Cross-Encoders evaluate full joint attention over query and document pairs, providing superior accuracy over Bi-Encoders."),
            Document(page_content="PyTorch allows direct tensor offloading to CUDA GPUs for accelerated transformer inference."),
            Document(page_content="Retrieval-Augmented Generation (RAG) reduces LLM hallucinations by supplying relevant grounding context."),
            Document(page_content="Self-attention mechanisms in standard Transformers scale quadratically O(N^2) with input sequence length.")
        ]

        # In-Memory FAISS Vector Index
        self.vector_store = FAISS.from_documents(knowledge_corpus, embeddings)

        # Stage 2: Cross-Encoder Reranker
        self.reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)
        print("[RAG Engine] Initialization complete.")

    def stage1_candidate_retrieval(self, query: str, top_k: int) -> List[Document]:
        """Stage 1: Quick Bi-Encoder Vector Search via FAISS."""
        return self.vector_store.similarity_search(query, k=top_k)

    def stage2_cross_encoder_rerank(self, query: str, candidate_docs: List[Document], top_n: int) -> List[str]:
        """Stage 2: Precision Cross-Encoder Reranking."""
        if not candidate_docs:
            return []

        # Prepare query-document pairs for scoring
        pairs = [[query, doc.page_content] for doc in candidate_docs]
        scores = self.reranker.predict(pairs)

        # Sort documents by cross-encoder relevance score
        scored_docs = sorted(zip(scores, candidate_docs), key=lambda x: x[0], reverse=True)
        return [doc.page_content for _, doc in scored_docs[:top_n]]


# Instantiate global RAG service
rag_service = AsyncRAGService()

In [12]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    # Initialize heavy models during server startup
    rag_service.initialize_models_and_index()
    yield
    print("[RAG Engine] Shutting down service...")

app = FastAPI(
    title="Async RAG Pipeline with Dual-Stage Reranking",
    description="High-throughput RAG engine using FastAPI, LangChain, FAISS & Cross-Encoders",
    version="1.0.0",
    lifespan=lifespan
)


# Request and Response Data Models
class RAGQueryRequest(BaseModel):
    query: str
    top_k_candidates: int = 5
    top_n_context: int = 2

class RAGQueryResponse(BaseModel):
    query: str
    retrieved_context: List[str]
    generated_answer: str
    total_latency_ms: float


# Simulated Async LLM Synthesis
async def async_llm_synthesis(query: str, context_chunks: List[str]) -> str:
    """Simulates real-time, non-blocking concurrent LLM text generation."""
    await asyncio.sleep(0.12)  # Non-blocking latency simulation
    joined_context = " | ".join(context_chunks) if context_chunks else "No relevant context found."
    return f"Synthesized Response: Based on context [{joined_context}], the answer to '{query}' is derived directly from context."

In [13]:
@app.post("/api/rag/query", response_model=RAGQueryResponse)
async def handle_rag_query(request: RAGQueryRequest):
    start_time = time.perf_counter()

    try:
        # Offload CPU/GPU PyTorch tasks to background threads to keep event loop free
        candidates = await asyncio.to_thread(
            rag_service.stage1_candidate_retrieval,
            request.query,
            request.top_k_candidates
        )

        reranked_context = await asyncio.to_thread(
            rag_service.stage2_cross_encoder_rerank,
            request.query,
            candidates,
            request.top_n_context
        )

        # Async non-blocking LLM synthesis call
        answer = await async_llm_synthesis(request.query, reranked_context)

        latency = (time.perf_counter() - start_time) * 1000.0

        return RAGQueryResponse(
            query=request.query,
            retrieved_context=reranked_context,
            generated_answer=answer,
            total_latency_ms=round(latency, 2)
        )

    except Exception as err:
        raise HTTPException(status_code=500, detail=str(err))

In [14]:
def run_service():
    """Runs FastAPI gracefully without triggering asyncio runner conflicts."""
    try:
        # If in a running event loop (e.g., Colab / Jupyter)
        loop = asyncio.get_running_loop()
        import nest_asyncio
        nest_asyncio.apply()
        config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
        server = uvicorn.Server(config)
        loop.create_task(server.serve())
        print("Server running in background loop at http://localhost:8000")
    except RuntimeError:
        # If running as a standard standalone Python script
        uvicorn.run(app, host="0.0.0.0", port=8000)

if __name__ == "__main__":
    run_service()

Server running in background loop at http://localhost:8000
